# SISTEMA DE BÚSQUEDA SEMÁNTICA
## Comparativa: Bag of Words (BoW) vs TF-IDF

**Autores:** Walter Reyes, Samuel Yambay, Franklin Acero, Karina Moreno, Patricio Gines, Kevin Samaniego, Bryan Caguana  
**Fecha:** 28 de marzo 2026  
**Curso:** 1  
**Grupo:** 3 - Representación Estadística (BoW y TF-IDF)

### Descripción
Este proyecto implementa dos métodos de representación de texto para búsqueda documental:
- **Bag of Words (BoW):** Representación basada en frecuencia de términos
- **TF-IDF:** Ponderación que considera importancia relativa de términos

### Objetivos
1. Cargar documentos de texto desde una carpeta
2. Indexar con ambos métodos (BoW y TF-IDF)
3. Comparar resultados de similitud coseno
4. Identificar diferencias entre ambos enfoques

## 📚 Importación de Librerías

In [2]:
# Librerías para procesamiento de texto y machine learning
import os
import numpy as np
import pandas as pd
from pathlib import Path

# Scikit-learn para vectorización y similitud
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

print("✅ Librerías importadas correctamente")

✅ Librerías importadas correctamente


## 📝 Descargar Archivos

In [3]:
# Descargar Archivos desde GitHub
import os
import subprocess

print("🔧 Configurando entorno en Colab...")

# Verificar si estamos en Colab
try:
    from google.colab import drive
    IN_COLAB = True
    print("✅ Google Colab detectado")
except:
    IN_COLAB = False
    print("✅ Ejecución local detectada")

if IN_COLAB:
    # Clonar el repositorio de GitHub
    print("📥 Descargando archivos desde GitHub...")
    
    # Verificar si ya existe la carpeta
    if not os.path.exists('documentos'):
        # Clonar el repo
        !git clone https://github.com/Bryan2oo/Busqueda.git temp_repo
        
        # Mover la carpeta documentos al directorio actual
        !mv temp_repo/documentos .
        
        # Limpiar
        !rm -rf temp_repo
        
        print("✅ Carpeta 'documentos' descargada exitosamente")
    else:
        print("✅ Carpeta 'documentos' ya existe")
    
    # Configurar ruta
    CARPETA_DOCUMENTOS = "documentos"
else:
    # En local, usar ruta relativa normal
    CARPETA_DOCUMENTOS = "documentos"

# Verificar archivos
print(f"\n📂 Buscando documentos en: {CARPETA_DOCUMENTOS}")
if os.path.exists(CARPETA_DOCUMENTOS):
    archivos = os.listdir(CARPETA_DOCUMENTOS)
    print(f"✅ {len(archivos)} archivos encontrados:")
    for archivo in archivos:
        print(f"   📄 {archivo}")
else:
    print(f"❌ Error: La carpeta '{CARPETA_DOCUMENTOS}' no existe")

🔧 Configurando entorno en Colab...
✅ Ejecución local detectada

📂 Buscando documentos en: documentos
✅ 4 archivos encontrados:
   📄 deportes.txt
   📄 mascotas.txt
   📄 salud.txt
   📄 Tecnologia.txt


## ⚙️ Configuración del Proyecto

In [4]:
# Configuración de rutas
SCRIPT_DIR = os.path.dirname(os.path.abspath('__file__'))
CARPETA_DOCUMENTOS = os.path.join(SCRIPT_DIR, "documentos")

print(f"📁 Ruta del script: {SCRIPT_DIR}")
print(f"📂 Buscando documentos en: {CARPETA_DOCUMENTOS}")

📁 Ruta del script: C:\Users\Bryan\Desktop\Busqueda
📂 Buscando documentos en: C:\Users\Bryan\Desktop\Busqueda\documentos


## 🛑 Stop Words en Español
Palabras que se eliminarán del análisis por ser muy comunes.

In [8]:
STOP_WORDS_ES = [
    'el', 'la', 'los', 'las', 'un', 'una', 'unos', 'unas',
    'de', 'del', 'al', 'en', 'con', 'por', 'para', 'sin', 'sobre',
    'y', 'o', 'pero', 'si', 'no', 'que', 'como', 'cuando', 'donde',
    'este', 'esta', 'estos', 'estas', 'ese', 'esa', 'esos', 'esas',
    'mi', 'tu', 'su', 'nuestro', 'vuestro',
    'me', 'te', 'se', 'nos', 'os',
    'es', 'son', 'era', 'eran', 'fue', 'fueron', 'ser', 'estar',
    'ha', 'han', 'he', 'hemos', 'haber',
    'lo', 'le', 'les', 'la', 'las', 'los'
]

print(f"✅ {len(STOP_WORDS_ES)} stop words configuradas")

✅ 63 stop words configuradas


## 📂 Función para Cargar Documentos

In [9]:
def cargar_documentos(ruta_carpeta):
    """
    Carga todos los archivos .txt de una carpeta.
    
    Args:
        ruta_carpeta (str): Ruta de la carpeta con documentos
    
    Returns:
        tuple: (lista de contenidos, lista de nombres de archivo)
    """
    documentos = []
    nombres_archivos = []
    ruta = Path(ruta_carpeta)

    if not ruta.exists():
        print(f"⚠️ La carpeta '{ruta_carpeta}' no existe")
        return documentos, nombres_archivos

    for archivo in sorted(ruta.glob("*.txt")):
        try:
            with open(archivo, 'r', encoding='utf-8') as f:
                contenido = f.read().strip()
                if contenido:
                    documentos.append(contenido)
                    nombres_archivos.append(archivo.name)
        except Exception as e:
            print(f"❌ Error leyendo {archivo}: {e}")

    return documentos, nombres_archivos

## 🤖 Clase Comparador BoW vs TF-IDF
Clase principal que implementa ambos métodos de vectorización.

In [25]:
class ComparadorBoW_TFIDF:
    def __init__(self, ruta_carpeta):
        self.ruta_carpeta = ruta_carpeta
        self.documentos = []
        self.nombres_archivos = []
        self.bow_vectorizer = None
        self.tfidf_vectorizer = None
        self.matriz_bow = None
        self.matriz_tfidf = None

    def indexar(self):
        """Indexa documentos con AMBOS métodos"""
        print("\n📂 Cargando documentos...")
        self.documentos, self.nombres_archivos = cargar_documentos(self.ruta_carpeta)

        if len(self.documentos) == 0:
            print(f"⚠️ Agrega archivos .txt a '{self.ruta_carpeta}'")
            return False

        print(f"✅ {len(self.documentos)} documentos encontrados")

        # Bag of Words
        print("\n⚙️ Creando matriz BoW (CountVectorizer)...")
        self.bow_vectorizer = CountVectorizer(
            stop_words=STOP_WORDS_ES,
            ngram_range=(1, 1)
        )
        self.matriz_bow = self.bow_vectorizer.fit_transform(self.documentos)

        # TF-IDF
        print("⚙️ Creando matriz TF-IDF (TfidfVectorizer)...")
        self.tfidf_vectorizer = TfidfVectorizer(
            stop_words=STOP_WORDS_ES,
            max_features=10000,
            ngram_range=(1, 1)
        )
        self.matriz_tfidf = self.tfidf_vectorizer.fit_transform(self.documentos)

        print("✅ Indexación completa")
        return True

    def mostrar_diferencias(self):
        """Muestra las diferencias entre BoW y TF-IDF"""
        print("\n" + "=" * 80)
        print("📊 COMPARATIVA: BAG OF WORDS vs TF-IDF")
        print("=" * 80)

        vocab_bow = self.bow_vectorizer.get_feature_names_out()
        vocab_tfidf = self.tfidf_vectorizer.get_feature_names_out()

        print(f"\n📏 Dimensiones:")
        print(f"   BoW:    {self.matriz_bow.shape}")
        print(f"   TF-IDF: {self.matriz_tfidf.shape}")

        print(f"\n🔢 MATRIZ BoW (primeros 5 términos):")
        print(f"   {vocab_bow[:5]}")
        print(self.matriz_bow.toarray()[:, :5])

        print(f"\n🔢 MATRIZ TF-IDF (primeros 5 términos):")
        print(f"   {vocab_tfidf[:5]}")
        print(np.round(self.matriz_tfidf.toarray()[:, :5], 4))

        print(f"\n📈 ESTADÍSTICAS:")
        print(f"   BoW - Máximo: {self.matriz_bow.max():.0f}, Mínimo: {self.matriz_bow.min():.0f}")
        print(f"   TF-IDF - Máximo: {self.matriz_tfidf.max():.4f}, Mínimo: {self.matriz_tfidf.min():.4f}")

        print("=" * 80)

    def buscar_comparativo(self, texto_consulta, top_n=3):
        """Busca con AMBOS métodos y muestra resultados comparativos"""
        print("\n" + "=" * 80)
        print(f"🔍 CONSULTA: '{texto_consulta}'")
        print("=" * 80)

        vector_bow = self.bow_vectorizer.transform([texto_consulta])
        vector_tfidf = self.tfidf_vectorizer.transform([texto_consulta])

        sim_bow = cosine_similarity(vector_bow, self.matriz_bow)[0]
        sim_tfidf = cosine_similarity(vector_tfidf, self.matriz_tfidf)[0]

        idx_bow = np.argsort(sim_bow)[::-1]
        idx_tfidf = np.argsort(sim_tfidf)[::-1]

        print("\n📄 MÉTODO 1: BAG OF WORDS (BoW)")
        print("-" * 80)
        print(f"{'POS':<5} | {'SIMILITUD':<12} | {'ARCHIVO':<25}")
        print("-" * 80)

        for i in range(min(top_n, len(idx_bow))):
            idx = idx_bow[i]
            if sim_bow[idx] > 0:
                print(f"{i + 1:<5} | {sim_bow[idx]:.4f}       | {self.nombres_archivos[idx]}")

        print("\n📄 MÉTODO 2: TF-IDF")
        print("-" * 80)
        print(f"{'POS':<5} | {'SIMILITUD':<12} | {'ARCHIVO':<25}")
        print("-" * 80)

        for i in range(min(top_n, len(idx_tfidf))):
            idx = idx_tfidf[i]
            if sim_tfidf[idx] > 0:
                print(f"{i + 1:<5} | {sim_tfidf[idx]:.4f}       | {self.nombres_archivos[idx]}")

        print("=" * 80)

        # Análisis de resultados
        mejor_bow = self.nombres_archivos[idx_bow[0]] if sim_bow[idx_bow[0]] > 0 else "Ninguno"
        mejor_tfidf = self.nombres_archivos[idx_tfidf[0]] if sim_tfidf[idx_tfidf[0]] > 0 else "Ninguno"

        print("\n💡 ANÁLISIS:")
        if mejor_bow == mejor_tfidf:
            print(f"   ✅ Ambos métodos coinciden: '{mejor_bow}'")
        else:
            print(f"   ⚠️ Resultados diferentes:")
            print(f"      BoW:    '{mejor_bow}'")
            print(f"      TF-IDF: '{mejor_tfidf}'")

        print("=" * 80)

## 🧪 Pruebas del Sistema
Ejecutamos el sistema con consultas de ejemplo.

In [26]:
# Inicializar el comparador
print("=" * 80)
print("📊 GRUPO 3: REPRESENTACIÓN ESTADÍSTICA")
print("   Comparativa: Bag of Words (BoW) vs TF-IDF")
print("=" * 80)

comparador = ComparadorBoW_TFIDF(CARPETA_DOCUMENTOS)

# Indexar documentos
if comparador.indexar():
    print("\n✅ Sistema listo para búsquedas")
else:
    print("\n❌ Error: No se pudieron cargar los documentos")

📊 GRUPO 3: REPRESENTACIÓN ESTADÍSTICA
   Comparativa: Bag of Words (BoW) vs TF-IDF

📂 Cargando documentos...
✅ 4 documentos encontrados

⚙️ Creando matriz BoW (CountVectorizer)...
⚙️ Creando matriz TF-IDF (TfidfVectorizer)...
✅ Indexación completa

✅ Sistema listo para búsquedas


In [27]:
# Mostrar comparativa de matrices
comparador.mostrar_diferencias()


📊 COMPARATIVA: BAG OF WORDS vs TF-IDF

📏 Dimensiones:
   BoW:    (4, 325)
   TF-IDF: (4, 325)

🔢 MATRIZ BoW (primeros 5 términos):
   ['000' '200' '2026' '252' '3d']
[[0 1 2 0 0]
 [1 0 2 1 0]
 [0 0 2 0 0]
 [0 0 1 0 1]]

🔢 MATRIZ TF-IDF (primeros 5 términos):
   ['000' '200' '2026' '252' '3d']
[[0.     0.087  0.0908 0.     0.    ]
 [0.0854 0.     0.0891 0.0854 0.    ]
 [0.     0.     0.0948 0.     0.    ]
 [0.     0.     0.045  0.     0.0861]]

📈 ESTADÍSTICAS:
   BoW - Máximo: 5, Mínimo: 0
   TF-IDF - Máximo: 0.4271, Mínimo: 0.0000


## 🔍 Ejemplos de Búsqueda
Prueba con diferentes términos relacionados a tus archivos.

In [28]:
# Ejemplo 1: Búsqueda sobre deportes
comparador.buscar_comparativo("futbol deporte campeonato", top_n=3)


🔍 CONSULTA: 'futbol deporte campeonato'

📄 MÉTODO 1: BAG OF WORDS (BoW)
--------------------------------------------------------------------------------
POS   | SIMILITUD    | ARCHIVO                  
--------------------------------------------------------------------------------
1     | 0.3266       | deportes.txt

📄 MÉTODO 2: TF-IDF
--------------------------------------------------------------------------------
POS   | SIMILITUD    | ARCHIVO                  
--------------------------------------------------------------------------------
1     | 0.3481       | deportes.txt

💡 ANÁLISIS:
   ✅ Ambos métodos coinciden: 'deportes.txt'


In [29]:
# Ejemplo 2: Búsqueda sobre mascotas
comparador.buscar_comparativo("perro gato mascota", top_n=3)


🔍 CONSULTA: 'perro gato mascota'

📄 MÉTODO 1: BAG OF WORDS (BoW)
--------------------------------------------------------------------------------
POS   | SIMILITUD    | ARCHIVO                  
--------------------------------------------------------------------------------

📄 MÉTODO 2: TF-IDF
--------------------------------------------------------------------------------
POS   | SIMILITUD    | ARCHIVO                  
--------------------------------------------------------------------------------

💡 ANÁLISIS:
   ✅ Ambos métodos coinciden: 'Ninguno'


In [30]:
# Ejemplo 3: Búsqueda sobre salud
comparador.buscar_comparativo("medicina salud enfermedad", top_n=3)


🔍 CONSULTA: 'medicina salud enfermedad'

📄 MÉTODO 1: BAG OF WORDS (BoW)
--------------------------------------------------------------------------------
POS   | SIMILITUD    | ARCHIVO                  
--------------------------------------------------------------------------------
1     | 0.4138       | salud.txt
2     | 0.2315       | mascotas.txt

📄 MÉTODO 2: TF-IDF
--------------------------------------------------------------------------------
POS   | SIMILITUD    | ARCHIVO                  
--------------------------------------------------------------------------------
1     | 0.3580       | salud.txt
2     | 0.2020       | mascotas.txt

💡 ANÁLISIS:
   ✅ Ambos métodos coinciden: 'salud.txt'


## 📈 Conclusiones

### Diferencias entre BoW y TF-IDF:

| Característica | BoW | TF-IDF |
|---------------|-----|--------|
| **Valores** | Frecuencias enteras | Ponderación 0-1 |
| **Palabras comunes** | Las cuenta igual | Las penaliza |
| **Palabras raras** | Mismo peso | Mayor peso |
| **Recomendado para** | Textos cortos | Búsqueda documental |

### ✅ Ventajas de este sistema:
- Compara dos métodos simultáneamente
- Usa similitud coseno para ranking
- Elimina stop words en español
- Muestra análisis de coincidencias
### 📊 Observaciones:
- TF-IDF tiende a dar mejores resultados en búsquedas documentales
- BoW es más simple pero puede ser afectado por palabras muy frecuentes
- La similitud coseno permite comparar documentos de diferentes longitudes